In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
import os

BASE_DIR = "/content/drive/MyDrive/JoshTalks_ASR"

RAW_DIR = os.path.join(BASE_DIR, "01_raw")
DOWN_AUDIO_DIR = os.path.join(BASE_DIR, "02_downloaded", "audio")
DOWN_TRANS_DIR = os.path.join(BASE_DIR, "02_downloaded", "transcripts")
DOWN_META_DIR = os.path.join(BASE_DIR, "02_downloaded", "metadata")

PROCESSED_DIR = os.path.join(BASE_DIR, "03_processed")
MANIFEST_DIR = os.path.join(PROCESSED_DIR, "manifests")
SPLIT_DIR = os.path.join(PROCESSED_DIR, "train_val_split")
DEBUG_DIR = os.path.join(PROCESSED_DIR, "debug_samples")

for path in [DOWN_AUDIO_DIR, DOWN_TRANS_DIR, DOWN_META_DIR, MANIFEST_DIR, SPLIT_DIR, DEBUG_DIR]:
    os.makedirs(path, exist_ok=True)

print("Folders ready.")
print("Base path:", BASE_DIR)

Folders ready.
Base path: /content/drive/MyDrive/JoshTalks_ASR


In [34]:
!pip install -q pandas requests librosa soundfile tqdm datasets transformers evaluate jiwer scikit-learn
!apt-get update -qq
!apt-get install -y -qq ffmpeg

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [35]:
import os
import json
import re
import pandas as pd
import requests
from tqdm import tqdm
import librosa
import soundfile as sf

In [36]:
csv_path = os.path.join(RAW_DIR, "FT Data - data.csv")
df = pd.read_csv(csv_path)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

df.head()

Shape: (104, 7)
Columns: ['user_id', 'recording_id', 'language', 'duration', 'rec_url_gcp', 'transcription_url_gcp', 'metadata_url_gcp']


,user_id,recording_id,language,duration,rec_url_gcp,transcription_url_gcp,metadata_url_gcp
0,245746,825780,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
1,291038,825727,hi,443,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
2,246004,988596,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
3,93626,990175,hi,475,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
4,286851,526266,hi,522,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...


In [37]:
print(df.isnull().sum())

user_id                  0
recording_id             0
language                 0
duration                 0
rec_url_gcp              0
transcription_url_gcp    0
metadata_url_gcp         0
dtype: int64


In [38]:
print("Language distribution:")
print(df["language"].value_counts())

print("\nDuration stats:")
print(df["duration"].describe())


Language distribution:
language
hi    104
Name: count, dtype: int64

Duration stats:
count     104.000000
mean      757.596154
std       274.708973
min       438.000000
25%       526.500000
50%       667.000000
75%      1080.000000
max      1194.000000
Name: duration, dtype: float64


In [39]:
debug_df = df.head(5).copy()

debug_path = os.path.join(DEBUG_DIR, "debug_5.csv")
debug_df.to_csv(debug_path, index=False)

print("Saved debug subset to:", debug_path)
print("Debug shape:", debug_df.shape)

debug_df[["recording_id", "rec_url_gcp", "transcription_url_gcp"]].head()

Saved debug subset to: /content/drive/MyDrive/JoshTalks_ASR/03_processed/debug_samples/debug_5.csv
Debug shape: (5, 7)


,recording_id,rec_url_gcp,transcription_url_gcp
0,825780,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
1,825727,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
2,988596,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
3,990175,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...
4,526266,https://storage.googleapis.com/joshtalks-data-...,https://storage.googleapis.com/joshtalks-data-...


In [40]:
for i in range(5):
    print(f"\nRow {i}")
    print("Recording ID:", debug_df.iloc[i]["recording_id"])
    print("Audio URL:")
    print(debug_df.iloc[i]["rec_url_gcp"])
    print("Transcript URL:")
    print(debug_df.iloc[i]["transcription_url_gcp"])


Row 0
Recording ID: 825780
Audio URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav
Transcript URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_transcription.json

Row 1
Recording ID: 825727
Audio URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825727_audio.wav
Transcript URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825727_transcription.json

Row 2
Recording ID: 988596
Audio URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/988596_audio.wav
Transcript URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/988596_transcription.json

Row 3
Recording ID: 990175
Audio URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/990175_audio.wav
Transcript URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/990175_transcription.json

Row

In [41]:
def download_json(url, save_path):
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()

        with open(save_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

        return True, data
    except Exception as e:
        return False, str(e)

In [42]:
test_turl = debug_df.iloc[0]["transcription_url_gcp"]
test_save_path = os.path.join(DOWN_TRANS_DIR, "test_0_transcription.json")

success, result = download_json(test_turl, test_save_path)

print("Success:", success)
print("Saved path:", test_save_path)

if success:
    print("Type of JSON:", type(result))
    if isinstance(result, dict):
        print("Keys:", list(result.keys()))
    elif isinstance(result, list):
        print("List length:", len(result))
        if len(result) > 0:
            print("First item:", result[0])
else:
    print("Error:", result)

Success: False
Saved path: /content/drive/MyDrive/JoshTalks_ASR/02_downloaded/transcripts/test_0_transcription.json
Error: 404 Client Error: Not Found for url: https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_transcription.json


In [43]:
for i in range(3):
    print(f"\nRow {i}")
    print("Recording ID:", debug_df.iloc[i]["recording_id"])
    print("Metadata URL:")
    print(debug_df.iloc[i]["metadata_url_gcp"])


Row 0
Recording ID: 825780
Metadata URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_metadata.json

Row 1
Recording ID: 825727
Metadata URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825727_metadata.json

Row 2
Recording ID: 988596
Metadata URL:
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/988596_metadata.json


In [44]:
test_murl = debug_df.iloc[0]["metadata_url_gcp"]
test_meta_save_path = os.path.join(DOWN_META_DIR, "test_0_metadata.json")

success, result = download_json(test_murl, test_meta_save_path)

print("Success:", success)
print("Saved path:", test_meta_save_path)

if success:
    print("Type of JSON:", type(result))
    if isinstance(result, dict):
        print("Keys:", list(result.keys()))
        print("\nFull JSON:")
        print(json.dumps(result, ensure_ascii=False, indent=2))
    elif isinstance(result, list):
        print("List length:", len(result))
        if len(result) > 0:
            print("First item:", result[0])
else:
    print("Error:", result)

Success: False
Saved path: /content/drive/MyDrive/JoshTalks_ASR/02_downloaded/metadata/test_0_metadata.json
Error: 404 Client Error: Not Found for url: https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_metadata.json


In [45]:
def download_file(url, save_path):
    try:
        response = requests.get(url, stream=True, timeout=60)
        response.raise_for_status()

        with open(save_path, "wb") as f:
            for chunk in response.iter_content(chunk_size=8192):
                if chunk:
                    f.write(chunk)

        return True, None
    except Exception as e:
        return False, str(e)

In [46]:
test_aurl = debug_df.iloc[0]["rec_url_gcp"]
test_audio_save_path = os.path.join(DOWN_AUDIO_DIR, "test_0_audio.wav")

success, error = download_file(test_aurl, test_audio_save_path)

print("Success:", success)
print("Saved path:", test_audio_save_path)

if success:
    print("File exists:", os.path.exists(test_audio_save_path))
    print("File size (bytes):", os.path.getsize(test_audio_save_path))
else:
    print("Error:", error)

Success: False
Saved path: /content/drive/MyDrive/JoshTalks_ASR/02_downloaded/audio/test_0_audio.wav
Error: 404 Client Error: Not Found for url: https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav


In [47]:
for col in ["rec_url_gcp", "transcription_url_gcp", "metadata_url_gcp"]:
    print(f"\n===== {col} =====")
    for i in range(5):
        print(debug_df.iloc[i][col])


===== rec_url_gcp =====
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825727_audio.wav
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/988596_audio.wav
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/990175_audio.wav
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/639950/526266_audio.wav

===== transcription_url_gcp =====
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_transcription.json
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825727_transcription.json
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/988596_transcription.json
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/990175_transcription.json
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/6399

In [48]:
import requests

test_urls = [
    "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav",
    "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_transcription.json",
    "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_metadata.json",
]

for url in test_urls:
    try:
        r = requests.head(url, allow_redirects=True, timeout=20)
        print(url)
        print("Status:", r.status_code)
        print("-" * 60)
    except Exception as e:
        print(url)
        print("Error:", e)
        print("-" * 60)

https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav
Status: 404
------------------------------------------------------------
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_transcription.json
Status: 404
------------------------------------------------------------
https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_metadata.json
Status: 404
------------------------------------------------------------


In [49]:
base_tests = [
    "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav",
    "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780.wav",
    "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/audio/825780_audio.wav",
    "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/transcripts/825780_transcription.json",
    "https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/metadata/825780_metadata.json",
    "https://storage.googleapis.com/joshtalks-data-collection/data/hi/967179/825780_audio.wav",
    "https://storage.googleapis.com/joshtalks-data-collection/data/hi/967179/825780_transcription.json",
    "https://storage.googleapis.com/joshtalks-data-collection/data/hi/967179/825780_metadata.json",
]

for url in base_tests:
    try:
        r = requests.head(url, allow_redirects=True, timeout=20)
        print(r.status_code, "->", url)
    except Exception as e:
        print("ERROR ->", url, "|", e)

404 -> https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav
404 -> https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780.wav
404 -> https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/audio/825780_audio.wav
404 -> https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/transcripts/825780_transcription.json
404 -> https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/metadata/825780_metadata.json
404 -> https://storage.googleapis.com/joshtalks-data-collection/data/hi/967179/825780_audio.wav
404 -> https://storage.googleapis.com/joshtalks-data-collection/data/hi/967179/825780_transcription.json
404 -> https://storage.googleapis.com/joshtalks-data-collection/data/hi/967179/825780_metadata.json


In [50]:
url_check_rows = []

for idx, row in debug_df.iterrows():
    rec_id = row["recording_id"]
    audio_url = row["rec_url_gcp"]
    trans_url = row["transcription_url_gcp"]
    meta_url = row["metadata_url_gcp"]

    def get_status(url):
        try:
            r = requests.head(url, allow_redirects=True, timeout=20)
            return r.status_code
        except Exception as e:
            return str(e)

    url_check_rows.append({
        "recording_id": rec_id,
        "audio_status": get_status(audio_url),
        "transcription_status": get_status(trans_url),
        "metadata_status": get_status(meta_url),
    })

url_check_df = pd.DataFrame(url_check_rows)
url_check_df

,recording_id,audio_status,transcription_status,metadata_status
0,825780,404,404,404
1,825727,404,404,404
2,988596,404,404,404
3,990175,404,404,404
4,526266,404,404,404


In [51]:
url_check_save_path = os.path.join(MANIFEST_DIR, "debug_url_status_check.csv")
url_check_df.to_csv(url_check_save_path, index=False)

print("Saved to:", url_check_save_path)

Saved to: /content/drive/MyDrive/JoshTalks_ASR/03_processed/manifests/debug_url_status_check.csv


In [52]:
def convert_to_upload_goai_url(old_url):
    """
    Example:
    old:
    https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_transcription.json

    new:
    https://storage.googleapis.com/upload_goai/967179/825780_transcription.json
    """
    parts = old_url.split("/")
    folder_id = parts[-2]
    file_name = parts[-1]
    new_url = f"https://storage.googleapis.com/upload_goai/{folder_id}/{file_name}"
    return new_url


for i in range(3):
    old_turl = debug_df.iloc[i]["transcription_url_gcp"]
    new_turl = convert_to_upload_goai_url(old_turl)

    print(f"\nRow {i}")
    print("Old:", old_turl)
    print("New:", new_turl)


Row 0
Old: https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_transcription.json
New: https://storage.googleapis.com/upload_goai/967179/825780_transcription.json

Row 1
Old: https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825727_transcription.json
New: https://storage.googleapis.com/upload_goai/967179/825727_transcription.json

Row 2
Old: https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/988596_transcription.json
New: https://storage.googleapis.com/upload_goai/1147542/988596_transcription.json


In [53]:
test_new_turl = convert_to_upload_goai_url(debug_df.iloc[0]["transcription_url_gcp"])
test_new_save_path = os.path.join(DOWN_TRANS_DIR, "test_0_transcription_fixed.json")

success, result = download_json(test_new_turl, test_new_save_path)

print("Converted URL:", test_new_turl)
print("Success:", success)
print("Saved path:", test_new_save_path)

if success:
    print("Type of JSON:", type(result))
    if isinstance(result, list):
        print("List length:", len(result))
        if len(result) > 0:
            print("First item keys:", result[0].keys() if isinstance(result[0], dict) else type(result[0]))
            print("First item:", result[0])
    elif isinstance(result, dict):
        print("Keys:", list(result.keys()))
else:
    print("Error:", result)

Converted URL: https://storage.googleapis.com/upload_goai/967179/825780_transcription.json
Success: True
Saved path: /content/drive/MyDrive/JoshTalks_ASR/02_downloaded/transcripts/test_0_transcription_fixed.json
Type of JSON: <class 'list'>
List length: 24
First item keys: dict_keys(['start', 'end', 'speaker_id', 'text'])
First item: {'start': 0.11, 'end': 14.42, 'speaker_id': 245746, 'text': 'अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब'}


In [54]:
def extract_transcript_text(json_data):
    texts = []

    if isinstance(json_data, list):
        for item in json_data:
            if isinstance(item, dict) and "text" in item:
                t = str(item["text"]).strip()
                if t:
                    texts.append(t)

    elif isinstance(json_data, dict):
        if "text" in json_data and isinstance(json_data["text"], str):
            texts.append(json_data["text"].strip())
        for key in ["segments", "chunks", "utterances"]:
            if key in json_data and isinstance(json_data[key], list):
                for item in json_data[key]:
                    if isinstance(item, dict) and "text" in item:
                        t = str(item["text"]).strip()
                        if t:
                            texts.append(t)

    return " ".join(texts).strip()

In [55]:
full_text = extract_transcript_text(result)

print("Transcript length:", len(full_text))
print("\nTranscript preview:\n")
print(full_text[:1500])

Transcript length: 2984

Transcript preview:

अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नहीं हो सकती थी तो हम वहां गया थे कुड़रमा घाटी तरफ पर दिवोग काफी जंगली एरिया है वह जो खांड जनजाति पाए जाती ना वहां पाए जाती है तो जंगल का सफर होता है जब हम रहने के लिए गए थे नातो चाहते के साथ जैसे हम वहाँ पहले एंटर किये थे तो पहले तो गिर गया थे लगड़ा के उपर से नीचे पहली बारी था क्योंकि चलना नहीं आता न वहाँ का जो लैंड एरिया होता है इधर उधर काफी इधर जाओ तो उधर लुढ़क जाओगे हां तो फिर वहां जो दिन भर तो दिन में तो खोजने में वक्त बीत गया। जब रात की बारी आई तो हमने टेंट गड़ा और रहा तो जब पता जैसी रात हुआ ना शाम मतलब छै सात में इतना अजीब सा आवाज आने लगा बहुत अजीब सा डर तो इतना लगा था कि अगर कोई आता तो हम टैंट उखाड़ ही बाग जाते लेकिन कुछ आया नहीं ऐसा कुछ हुआ नहीं डर लगता है लेकिन बहुत सा

In [56]:
for i in range(3):
    old_aurl = debug_df.iloc[i]["rec_url_gcp"]
    new_aurl = convert_to_upload_goai_url(old_aurl)

    print(f"\nRow {i}")
    print("Old:", old_aurl)
    print("New:", new_aurl)


Row 0
Old: https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825780_audio.wav
New: https://storage.googleapis.com/upload_goai/967179/825780_audio.wav

Row 1
Old: https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/967179/825727_audio.wav
New: https://storage.googleapis.com/upload_goai/967179/825727_audio.wav

Row 2
Old: https://storage.googleapis.com/joshtalks-data-collection/hq_data/hi/1147542/988596_audio.wav
New: https://storage.googleapis.com/upload_goai/1147542/988596_audio.wav


In [57]:
test_new_aurl = convert_to_upload_goai_url(debug_df.iloc[0]["rec_url_gcp"])
test_audio_fixed_path = os.path.join(DOWN_AUDIO_DIR, "test_0_audio_fixed.wav")

success, error = download_file(test_new_aurl, test_audio_fixed_path)

print("Converted URL:", test_new_aurl)
print("Success:", success)
print("Saved path:", test_audio_fixed_path)

if success:
    print("File exists:", os.path.exists(test_audio_fixed_path))
    print("File size (bytes):", os.path.getsize(test_audio_fixed_path))
else:
    print("Error:", error)

Converted URL: https://storage.googleapis.com/upload_goai/967179/825780_audio.wav
Success: True
Saved path: /content/drive/MyDrive/JoshTalks_ASR/02_downloaded/audio/test_0_audio_fixed.wav
File exists: True
File size (bytes): 34307062


In [58]:
def convert_to_16k_mono(input_path, output_path):
    try:
        audio, sr = librosa.load(input_path, sr=16000, mono=True)
        sf.write(output_path, audio, 16000)
        return True, None
    except Exception as e:
        return False, str(e)

In [59]:
converted_audio_path = os.path.join(DOWN_AUDIO_DIR, "test_0_audio_fixed_16k.wav")

success, error = convert_to_16k_mono(test_audio_fixed_path, converted_audio_path)

print("Success:", success)
print("Converted path:", converted_audio_path)

if success:
    print("File exists:", os.path.exists(converted_audio_path))
    print("File size (bytes):", os.path.getsize(converted_audio_path))
else:
    print("Error:", error)

Success: True
Converted path: /content/drive/MyDrive/JoshTalks_ASR/02_downloaded/audio/test_0_audio_fixed_16k.wav
File exists: True
File size (bytes): 12447036


In [60]:
def process_one_row(row):
    try:
        recording_id = row["recording_id"]
        user_id = row["user_id"]
        duration = row["duration"]

        # ---------- transcript ----------
        transcript_url = convert_to_upload_goai_url(row["transcription_url_gcp"])
        transcript_save_path = os.path.join(DOWN_TRANS_DIR, f"{recording_id}_transcription.json")

        success_t, transcript_json = download_json(transcript_url, transcript_save_path)
        if not success_t:
            return {"recording_id": recording_id, "status": "failed_transcript_download", "error": str(transcript_json)}

        transcript_text = extract_transcript_text(transcript_json)
        if not transcript_text.strip():
            return {"recording_id": recording_id, "status": "failed_empty_transcript", "error": "Transcript text empty"}

        # ---------- audio ----------
        audio_url = convert_to_upload_goai_url(row["rec_url_gcp"])
        raw_audio_path = os.path.join(DOWN_AUDIO_DIR, f"{recording_id}_raw.wav")
        final_audio_path = os.path.join(DOWN_AUDIO_DIR, f"{recording_id}.wav")

        success_a, audio_error = download_file(audio_url, raw_audio_path)
        if not success_a:
            return {"recording_id": recording_id, "status": "failed_audio_download", "error": str(audio_error)}

        success_c, convert_error = convert_to_16k_mono(raw_audio_path, final_audio_path)
        if not success_c:
            return {"recording_id": recording_id, "status": "failed_audio_convert", "error": str(convert_error)}

        return {
            "recording_id": recording_id,
            "user_id": user_id,
            "duration": duration,
            "audio_path": final_audio_path,
            "text": transcript_text,
            "status": "success"
        }

    except Exception as e:
        return {"recording_id": row.get("recording_id", "unknown"), "status": "failed_exception", "error": str(e)}

In [61]:
one_result = process_one_row(debug_df.iloc[0])

print(type(one_result))
print(one_result.keys())
print("\nResult:\n")
print(one_result)

<class 'dict'>
dict_keys(['recording_id', 'user_id', 'duration', 'audio_path', 'text', 'status'])

Result:

{'recording_id': np.int64(825780), 'user_id': np.int64(245746), 'duration': np.int64(443), 'audio_path': '/content/drive/MyDrive/JoshTalks_ASR/02_downloaded/audio/825780.wav', 'text': 'अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बहुत कम दी जा रही है तो हमें उनको देखना था तो एक देखना था मतलब वो तो देखना था लेकिन हमारा प्रोजेक्ट भी था कि जो जन जाती पाई जाती है उधर कि उधर की एरिया में उसके बारे में देखना अब अनुभव करके कुछ लिखना था तो वह तो बिना देखिए नहीं हो सकती थी तो हम वहां गया थे कुड़रमा घाटी तरफ पर दिवोग काफी जंगली एरिया है वह जो खांड जनजाति पाए जाती ना वहां पाए जाती है तो जंगल का सफर होता है जब हम रहने के लिए गए थे नातो चाहते के साथ जैसे हम वहाँ पहले एंटर किये थे तो पहले तो गिर गया थे लगड़ा के उपर से नीचे पहली बारी था क्योंकि चलना नहीं आता न वहाँ का जो लैंड एरिया होता है इधर उधर काफी इधर जाओ तो उधर लुढ़क जाओगे हां तो फिर वहां जो दिन भर तो दिन में तो खोजने में वक्त बीत गया। जब 

In [62]:
debug_results = []

for idx, row in debug_df.iterrows():
    result = process_one_row(row)
    debug_results.append(result)
    print(f"Row {idx} -> {result['status']}")

debug_results_df = pd.DataFrame(debug_results)
debug_results_df

Row 0 -> success
Row 1 -> success
Row 2 -> success
Row 3 -> success
Row 4 -> success


,recording_id,user_id,duration,audio_path,text,status
0,825780,245746,443,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...,success
1,825727,291038,443,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,जी जी जी जी जी । जी जी जी हां उधर हां जी हा हा...,success
2,988596,246004,475,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,लेकिन हम लोग इसे छुपछुप के लोगों के घर जाकर खे...,success
3,990175,93626,475,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,जी जी जी जी जी मेरे तो जैसे बहुत सारी यादे हैं...,success
4,526266,286851,522,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,हां जी पहले बात करते हैं विवाह की तो इस मुवी म...,success


In [63]:
debug_results_save_path = os.path.join(MANIFEST_DIR, "processed_debug_5.csv")
debug_results_df.to_csv(debug_results_save_path, index=False)

print("Saved debug results to:", debug_results_save_path)
print("Successful rows:", (debug_results_df["status"] == "success").sum())

Saved debug results to: /content/drive/MyDrive/JoshTalks_ASR/03_processed/manifests/processed_debug_5.csv
Successful rows: 5


In [64]:
all_results = []

for idx, row in tqdm(df.iterrows(), total=len(df)):
    result = process_one_row(row)
    all_results.append(result)

all_results_df = pd.DataFrame(all_results)

print("Total rows processed:", len(all_results_df))
print("\nStatus counts:")
print(all_results_df["status"].value_counts())

all_results_df.head()

100%|██████████| 104/104 [17:33<00:00, 10.13s/it]

Total rows processed: 104

Status counts:
status
success    104
Name: count, dtype: int64


,recording_id,user_id,duration,audio_path,text,status
0,825780,245746,443,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...,success
1,825727,291038,443,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,जी जी जी जी जी । जी जी जी हां उधर हां जी हा हा...,success
2,988596,246004,475,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,लेकिन हम लोग इसे छुपछुप के लोगों के घर जाकर खे...,success
3,990175,93626,475,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,जी जी जी जी जी मेरे तो जैसे बहुत सारी यादे हैं...,success
4,526266,286851,522,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,हां जी पहले बात करते हैं विवाह की तो इस मुवी म...,success


In [65]:
cleaned_full_path = os.path.join(MANIFEST_DIR, "cleaned_dataset.csv")
all_results_df.to_csv(cleaned_full_path, index=False)

print("Saved full dataset to:", cleaned_full_path)
print("Total usable rows:", len(all_results_df))

Saved full dataset to: /content/drive/MyDrive/JoshTalks_ASR/03_processed/manifests/cleaned_dataset.csv
Total usable rows: 104


In [66]:
import os

audio_dir = DOWN_AUDIO_DIR

all_files = os.listdir(audio_dir)

raw_files = [f for f in all_files if f.endswith("_raw.wav")]
test_files = [f for f in all_files if f.startswith("test_")]
final_files = [f for f in all_files if f.endswith(".wav") and not f.endswith("_raw.wav") and not f.startswith("test_")]

print("Raw files:", len(raw_files))
print("Test files:", len(test_files))
print("Final files:", len(final_files))

Raw files: 104
Test files: 2
Final files: 104


In [67]:
deleted_count = 0

for f in raw_files + test_files:
    file_path = os.path.join(audio_dir, f)
    if os.path.exists(file_path):
        os.remove(file_path)
        deleted_count += 1

print("Deleted files:", deleted_count)

Deleted files: 106


In [68]:
all_files = os.listdir(audio_dir)

raw_files = [f for f in all_files if f.endswith("_raw.wav")]
test_files = [f for f in all_files if f.startswith("test_")]
final_files = [f for f in all_files if f.endswith(".wav") and not f.endswith("_raw.wav") and not f.startswith("test_")]

print("Raw files left:", len(raw_files))
print("Test files left:", len(test_files))
print("Final files left:", len(final_files))

Raw files left: 0
Test files left: 0
Final files left: 104


In [69]:
final_df = all_results_df[all_results_df["status"] == "success"].copy()

final_df = final_df[["recording_id", "user_id", "duration", "audio_path", "text"]]

final_manifest_path = os.path.join(MANIFEST_DIR, "final_manifest.csv")
final_df.to_csv(final_manifest_path, index=False)

print("Saved final manifest to:", final_manifest_path)
print("Final rows:", len(final_df))
final_df.head()

Saved final manifest to: /content/drive/MyDrive/JoshTalks_ASR/03_processed/manifests/final_manifest.csv
Final rows: 104


,recording_id,user_id,duration,audio_path,text
0,825780,245746,443,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,अब काफी अच्छा होता है क्योंकि उनकी जनसंख्या बह...
1,825727,291038,443,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,जी जी जी जी जी । जी जी जी हां उधर हां जी हा हा...
2,988596,246004,475,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,लेकिन हम लोग इसे छुपछुप के लोगों के घर जाकर खे...
3,990175,93626,475,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,जी जी जी जी जी मेरे तो जैसे बहुत सारी यादे हैं...
4,526266,286851,522,/content/drive/MyDrive/JoshTalks_ASR/02_downlo...,हां जी पहले बात करते हैं विवाह की तो इस मुवी म...


In [70]:
from sklearn.model_selection import train_test_split

train_df, val_df = train_test_split(
    final_df,
    test_size=0.1,
    random_state=42,
    shuffle=True
)

train_path = os.path.join(SPLIT_DIR, "train.csv")
val_path = os.path.join(SPLIT_DIR, "val.csv")

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Saved train to:", train_path)
print("Saved val to:", val_path)

Train rows: 93
Val rows: 11
Saved train to: /content/drive/MyDrive/JoshTalks_ASR/03_processed/train_val_split/train.csv
Saved val to: /content/drive/MyDrive/JoshTalks_ASR/03_processed/train_val_split/val.csv
